# Load Models

In the previous notebook we saved the large model. In this notebook we have a look on how to reload it.

## Setup & Data

In [11]:
# Import packages
import os
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

tf.keras.backend.set_floatx('float64')

# Define random seed for whole notebook
RSEED=42

In [15]:
# Load data
df = pd.read_csv("data/boston.csv")

# Define target 
#y = df.pop("MEDV")

# Split into train and test set
#X_train, X_test, y_train, y_test = train_test_split(df, y, random_state=RSEED)

X = df.drop('MEDV', axis=1)   # still a DataFrame
y = df['MEDV']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RSEED)


In [16]:
print(df.shape)
print(y.shape)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(506, 13)
(506,)
(379, 12)
(127, 12)
(379,)
(127,)


In [17]:
print(type(X_train))   # likely <class 'numpy.ndarray'>
print(type(X_test))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>


In [18]:
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3.0,222.0,18.7,5.33,36.2


In [19]:
# Scale numerical features
# Scale numerical values
col_scale = ['CRIM', 'ZN', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'TAX', 'PTRATIO', 'LSTAT']

scaler = MinMaxScaler()
X_train[col_scale] = scaler.fit_transform(X_train[col_scale])
X_test[col_scale] = scaler.transform(X_test[col_scale])

# Convert to np array
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

The SavedModel format is a directory containing a protobuf binary and a TensorFlow checkpoint. Inspect the saved model directory:

In [21]:
ls saved_model

my_large_model/


In [22]:
ls saved_model/my_large_model

assets/            keras_metadata.pb  saved_model.pb     variables/


## SavedModel format

Reload a fresh Keras model from the saved model:

In [23]:
# Load the saved model
with tf.device('/cpu:0'):
    new_large_model = tf.keras.models.load_model('saved_model/my_large_model')

# Check its architecture
new_large_model.summary()

Metal device set to: Apple M3

systemMemory: 16.00 GB
maxCacheSize: 5.92 GB



2026-06-02 00:59:52.635395: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-06-02 00:59:52.635770: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_11 (Dense)            (None, 512)               6656      
                                                                 
 dense_12 (Dense)            (None, 512)               262656    
                                                                 
 dense_13 (Dense)            (None, 512)               262656    
                                                                 
 dense_14 (Dense)            (None, 512)               262656    
                                                                 
 dense_15 (Dense)            (None, 1)                 513       
                                                                 
Total params: 795,137
Trainable params: 795,137
Non-trainable params: 0
_________________________________________________________________


The restored model is compiled with the same arguments as the original model. Try running evaluate and predict with the loaded model:

In [24]:
# Evaluate the restored model
with tf.device('/cpu:0'):
    loss, mse = new_large_model.evaluate(X_test, y_test, verbose=2)
print(f'Model MSE: {mse}')

4/4 - 0s - loss: 10.6102 - mse: 158.5622 - 73ms/epoch - 18ms/step
Model MSE: 158.56220012873433


2026-06-02 01:00:08.813113: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
